In [6]:
!pip install -q transformers accelerate qwen-vl-utils flash-attn --no-build-isolation

In [7]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

# Modelin adı
model_id = "Qwen/Qwen2-VL-7B-Instruct"

# Modeli A100'e uygun ayarlarla yüklüyoruz
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16, # Bellek tasarrufu ve performans dengesi
    attn_implementation="flash_attention_2", # A100 hızı için kritik
    device_map="auto", # Modelin GPU belleğine otomatik yerleşimi
)

# Resim ve metni hazırlayacak olan işlemci
processor = AutoProcessor.from_pretrained(model_id)

print("✅ Mühendis (7B) masasına oturdu ve çalışmaya hazır!")

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Mühendis (7B) masasına oturdu ve çalışmaya hazır!


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
from PIL import Image
from qwen_vl_utils import process_vision_info
import torch

def derin_besin_tespiti(image_path):
    # Modelin resmi en yüksek detayda görmesini sağlamak için işlemci ayarı
    # Resim çok küçük parçalara bölünerek (patch) analiz edilecek
    prompt = """
    Sen bir mikroskopik besin analiz uzmanısın. Resme en yüksek çözünürlükte bak.

    Görevlerin:
    1. Resimdeki her bir nesneyi tek tek tara.
    2. Her bir nesne için şunları yaz:
       - Kesin İsim
       - Adet (Eğer birden fazlaysa)
       - Koordinat: [ymin, xmin, ymax, xmax] (0-1000 arası hassas değerler)

    Dikkat: Dokulara ve renklere odaklan besinleri birbirleri ile karıştırma.
    """

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": prompt}
        ]
    }]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)

    # Görüntüyü parçalara ayırırken modelin limitlerini zorluyoruz (Sağlam sonuç için)
    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=1024, # Liste uzun olacağı için token limitini artırdık
            do_sample=False,      # En kesin (greedy) sonucu istiyoruz
            repetition_penalty=1.1
        )

    output = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return output.split("assistant\n")[-1]

# TEST
print(derin_besin_tespiti("/content/drive/MyDrive/Yemek_denemesi/kafe.jpg"))

Resimdeki nesneler:

1. Domates: 1 adet, [350, 40, 650, 280]
2. Zeytin: 7 adet, [700, 100, 900, 300]
3. Yeşil biber: 1 adet, [400, 10, 600, 200]
4. Kavun: 1 adet, [100, 100, 300, 300]
5. Sarımsak: 1 adet, [100, 300, 300, 500]
6. Tuzlu peynir: 1 adet, [700, 500, 900, 700]
7. Süt peyniri: 1 adet, [100, 500, 300, 700]
8. Portakal sarısı süt: 1 adet, [100, 300, 300, 500]
9. Kırmızı süt: 1 adet, [100, 300, 300, 500]
10. Sarımsaklı krem: 1 adet, [100, 300, 300, 500]

Not: Bu bilgiler resimdeki nesnelerin kesin türlerini belirtmek için kullanılmıştır. Besinlerin birbirleriyle karıştığına dair bir açıklama yapmak mümkün değil.


In [ ]:
import json
import difflib
import re
import torch
import gradio as gr
from qwen_vl_utils import process_vision_info

# --- 1. VERİTABANI YÜKLEME ---
JSON_PATH = "/content/drive/MyDrive/YemekVerisi/food_nutrition.json"
try:
    with open(JSON_PATH, 'r', encoding='utf-8') as f:
        besin_db = json.load(f)
    print("✅ Veritabanı başarıyla yüklendi.")
except FileNotFoundError:
    besin_db = []
    print("❌ HATA: JSON dosyası bulunamadı!")

# --- 2. YARDIMCI FONKSİYONLAR ---
def find_ingredient_with_meta(target_name):
    if not besin_db: return None
    all_candidates = []
    for idx, dish in enumerate(besin_db):
        if 'ingredients' in dish:
            for ingr in dish['ingredients']:
                item = ingr.copy()
                all_candidates.append(item)
    names = [i['name'].lower() for i in all_candidates]
    matches = difflib.get_close_matches(target_name.lower(), names, n=1, cutoff=0.5)
    if matches:
        for i in all_candidates:
            if i['name'].lower() == matches[0]: return i
    return None

# --- 3. ANALİZ MOTORU ---
def full_ai_nutrition_analyzer(image_path):
    if not image_path: return {"status": "error", "message": "Resim yok."}
    json_report = {"status": "success", "plate_analysis": [], "total_macros": {"calories": 0.0, "protein": 0.0, "carb": 0.0, "fat": 0.0}}

    prompt = "Identify all food and drink items in the image. Estimate their weight in grams. Output ONLY: - [Food]: [Weight]g"

    messages = [{"role": "user", "content": [{"type": "image", "image": image_path}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, padding=True, return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)

    raw_output = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].split("assistant\n")[-1]
    items_found = re.findall(r"-\s*\[?([^\]:]+)\]?:\s*(\d+)", raw_output)

    for name, grams in items_found:
        total_grams = float(grams)
        ingr_data = find_ingredient_with_meta(name.strip())
        item_entry = {"name": name.strip(), "detected_weight": f"{total_grams}g", "db_match": None}

        if ingr_data:
            ratio = total_grams / float(ingr_data.get('grams', 100))
            item_entry["db_match"] = {
                "db_name": ingr_data['name'],
                "calories": round(ingr_data.get('calories', 0) * ratio, 1),
                "nutrition": {
                    "protein": round(ingr_data.get('protein', 0) * ratio, 2),
                    "carb": round(ingr_data.get('carb', 0) * ratio, 2),
                    "fat": round(ingr_data.get('fat', 0) * ratio, 2)
                }
            }
            for k in json_report["total_macros"]:
                json_report["total_macros"][k] += item_entry["db_match"]["calories"] if k == "calories" else item_entry["db_match"]["nutrition"][k]

        json_report["plate_analysis"].append(item_entry)

    for key in json_report["total_macros"]: json_report["total_macros"][key] = round(json_report["total_macros"][key], 2)
    return json_report

# --- 4. GRADIO İÇİN SARMALAYICI FONKSİYONLAR ---
# Eksik olan 'gradio_fn' burada tanımlandı
def gradio_fn(img):
    return full_ai_nutrition_analyzer(img)

DIETITIAN_PROMPT = "Sen uzman bir diyetisyensin. Sadece beslenme ve sağlıklı yaşam konularında rehberlik et."

def dietitian_chat(message, history):
    messages = [{"role": "system", "content": DIETITIAN_PROMPT}]
    for val in history:
        messages.append({"role": "user", "content": val[0]})
        messages.append({"role": "assistant", "content": val[1]})
    messages.append({"role": "user", "content": message})

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=512)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0].split("assistant\n")[-1]

# --- 5. GRADIO ARAYÜZÜ ---
# Tema uyarısını gidermek için theme parametresini Blocks içinde bırakıyoruz (Gradio 5 için hala geçerli)
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🍎 Evrensel Besin Dedektörü & Diyetisyen Asistanı")

    with gr.Tabs():
        with gr.TabItem("📸 Görsel Analiz"):
            with gr.Row():
                with gr.Column():
                    im_in = gr.Image(type="filepath", label="Yemek Fotoğrafı")
                    btn = gr.Button("Analiz Et", variant="primary")
                with gr.Column():
                    json_out = gr.JSON(label="Analiz Sonucu")

            # Artık gradio_fn tanımlı olduğu için hata vermeyecek
            btn.click(gradio_fn, inputs=im_in, outputs=json_out, api_name="analyze")

        with gr.TabItem("💬 Diyetisyen Sohbet"):
            gr.ChatInterface(fn=dietitian_chat, api_name="chat")

demo.launch(share=True, debug=True)

✅ Veritabanı başarıyla yüklendi.


/tmp/ipython-input-3703355820.py:98: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://55af7ad0b4444ef344.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
